# Crawling web PTA UTM

In [1]:
!pip install builtwith

  DEPRECATION: Building 'builtwith' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'builtwith'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36092 sha256=3a705c4ff2a4c3e64064243f104415d020a0ab5a6f5fc0bddcc216bbf6d7afc2
  Stored in directory: c:\users\fedi arta\appdata\local\pip\cache\wheels\6e\bf\03\4e45fb3049b99c21360499dfdad979d11b73e972fb2d3ad56c
Successfully built builtwith


In [2]:
import builtwith

# Analisis teknologi yang digunakan
res = builtwith.parse('https://pta.trunojoyo.ac.id')
print(res)

{'web-servers': ['Nginx'], 'javascript-frameworks': ['jQuery', 'jQuery UI']}


## Crawling Link data text

In [3]:
import requests
from bs4 import BeautifulSoup

def crawl_website(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes

        soup = BeautifulSoup(response.content, 'html.parser')

        # Ambil semua judul h1, h2, h3
        headings = soup.find_all(['h1', 'h2', 'h3'])
        for heading in headings:
            print(f"{heading.name}: {heading.get_text()}")

        # Ambil semua link
        links = soup.find_all('a', href=True)
        for link in links:
            print(f"URL: {link['href']} | Teks: {link.get_text()}")

    except requests.exceptions.RequestException as e:
        print(f"Terjadi kesalahan saat mengakses {url}: {e}")

# Gunakan fungsi
crawl_website("https://pta.trunojoyo.ac.id")

h2: Daftar Karya Ilmiah
URL: index.html | Teks: 
URL: # | Teks: 14677Journal
URL: https://pta.trunojoyo.ac.id/ | Teks: Beranda
URL: https://pta.trunojoyo.ac.id/c_search/ | Teks: Pencarian
URL: https://pta.trunojoyo.ac.id/c_template/ | Teks: Download
URL: https://library.trunojoyo.ac.id/detil.php?id=23 | Teks: Petunjuk Upload
URL: https://pta.trunojoyo.ac.id/c_contact/ | Teks: Kontak
URL: # | Teks: STRATEGI PENGEMBANGAN MAKANAN DAN MINUMAN KHAS PULAU GILIGENTING GUNA MENDUKUNG PARIWISATA BERKELANJUTAN
URL: https://pta.trunojoyo.ac.id/welcome/detail/170361100003 | Teks: Selengkapnya
URL: # | Teks: PERUMUSAN SANKSI PIDANA BAGI MASYARAKAT SEKITAR HUTAN YANG MELAKUKAN PENCURIAN KAYU MILIK NEGARA DALAM UNDANG-UNDANG NOMOR 18 TAHUN 2013
URL: https://pta.trunojoyo.ac.id/welcome/detail/170111100053 | Teks: Selengkapnya
URL: # | Teks: Peran Teor Motivasi Herzberg Sebagai Mediator Self Efficacy, Lingkungan Kerja Dalam Meningkatkan Prestasi Kerja Pegawai ( Kantor Jasa Penilai Publik Guntur Eki And

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import json

def ptaa(
    max_page=5000,
    delay=1,
    save_path="C:\\Users\\Fedi Arta\\crawling_pta\\pta.csv",
    progress_path="C:\\Users\\Fedi Arta\\crawling_pta\\progress.json"
):
   
    data = {
        "penulis": [],
        "judul": [],
        "pembimbing_pertama": [],
        "pembimbing_kedua": [],
        "abstrak_id": [],
        "abstrak_en": [],
        "prodi": [],
        "url": []
    }

    # Buat folder kalau belum ada
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    prodis = {
        1: "Ilmu Hukum",
        24: "Magister Ilmu Hukum",
        2: "Teknologi Industri Pertanian",
        3: "Agribisnis",
        4: "Agroteknologi",
        5: "Ilmu Kelautan",
        35: "Manajemen Sumberdaya Perairan",
        37: "Magister PSDAL",
        6: "Ekonomi Pembangunan",
        7: "Manajemen",
        8: "Akuntansi",
        21: "D3 Akuntansi",
        22: "Magister Manajemen",
        25: "Magister Akuntansi",
        26: "D3 Enterpreneurship",
        36: "Magister Ilmu Ekonomi",
        41: "Doktor Ilmu Manajemen",
        9: "Teknik Industri",
        10: "Teknik Informatika",
        11: "Manajemen Informatika",
        19: "Teknik Multimedia dan Jaringan",
        20: "Mekatronika",
        23: "Teknik Elektro",
        31: "Sistem Informasi",
        32: "Teknik Mesin",
        33: "Teknik Mekatronika",
        12: "Sosiologi",
        13: "Ilmu Komunikasi",
        14: "Psikologi",
        15: "Sastra Inggris",
        16: "Ekonomi Syariah",
        17: "Hukum Bisnis Syariah",
        18: "PGSD",
        27: "Pendidikan Bhs & Sastra Indonesia",
        28: "Pendidikan Informatika",
        29: "Pendidikan IPA",
        30: "PGPAUD",
        38: "Pendidikan Profesi Guru",
        39: "Magister Pendidikan Dasar",
        40: "Doktor PSDAL",
        99: "MBKM",
        100: "Pascasarjana"
    }

    # Load progress jika ada
    if os.path.exists(progress_path):
        with open(progress_path, "r") as f:
            progress = json.load(f)
    else:
        progress = {}

    # Load data lama jika ada
    if os.path.exists(save_path):
        df_old = pd.read_csv(save_path)
        for key in data:
            data[key] = df_old[key].tolist()
    else:
        df_old = None

    try:
        for prodi_id, prodi_name in prodis.items():
            print(f"\n=== Mulai scrape prodi {prodi_name} (id={prodi_id}) ===")

            # Mulai dari halaman terakhir jika ada progress
            start_page = progress.get(str(prodi_id), 1)

            for page in range(start_page, max_page+1):
                url = f"https://pta.trunojoyo.ac.id/c_search/byprod/{prodi_id}/{page}"
                print(f"[{prodi_name}] Scraping halaman {page} -> {url}")

                r = requests.get(url)
                soup = BeautifulSoup(r.content, "html.parser")
                jurnals = soup.select('li[data-cat="#luxury"]')

                if not jurnals:
                    print(f"[{prodi_name}] Habis di halaman {page-1}")
                    break

                for jurnal in jurnals:
                    detail_url = jurnal.select_one('a.gray.button')['href']
                    detail_res = requests.get(detail_url)
                    soup1 = BeautifulSoup(detail_res.content, "html.parser")

                    isi = soup1.select_one('div#content_journal')

                    judul = isi.select_one('a.title').text.strip()

                    penulis = isi.find('span', string=lambda t: t and "Penulis" in t).text.split(' : ')[1].strip()
                    pembimbing_pertama = isi.find('span', string=lambda t: t and "Dosen Pembimbing I" in t).text.split(' : ')[1].strip()
                    pembimbing_kedua = isi.find('span', string=lambda t: t and "Dosen Pembimbing II" in t).text.split(' :')[1].strip()

                    abstrak_paragraphs = isi.find_all('p', align="justify")
                    abstrak_id = abstrak_paragraphs[0].get_text(strip=True) if len(abstrak_paragraphs) > 0 else ""
                    abstrak_en = abstrak_paragraphs[1].get_text(strip=True) if len(abstrak_paragraphs) > 1 else ""

                    data["penulis"].append(penulis)
                    data["judul"].append(judul)
                    data["pembimbing_pertama"].append(pembimbing_pertama)
                    data["pembimbing_kedua"].append(pembimbing_kedua)
                    data["abstrak_id"].append(abstrak_id)
                    data["abstrak_en"].append(abstrak_en)
                    data["prodi"].append(prodi_name)
                    data["url"].append(detail_url)

                    time.sleep(delay)

                # Simpan data ke CSV setiap selesai 1 halaman
                df = pd.DataFrame(data)
                df.to_csv(save_path, index=False, encoding="utf-8-sig")

                # Simpan progress
                progress[str(prodi_id)] = page + 1
                with open(progress_path, "w") as f:
                    json.dump(progress, f)

    except KeyboardInterrupt:
        print("\nScraping dihentikan oleh user. Progress disimpan.")
    except Exception as e:
        print(f"\nTerjadi error: {e}. Progress disimpan.")
    finally:
        # Simpan data terakhir dan progress
        df = pd.DataFrame(data)
        df.to_csv(save_path, index=False, encoding="utf-8-sig")
        with open(progress_path, "w") as f:
            json.dump(progress, f)
        print(f"\nData terakhir berhasil disimpan di {save_path}")
        print(f"Progress scraping disimpan di {progress_path}")

    return pd.DataFrame(data)

In [7]:
ptaa()


=== Mulai scrape prodi Ilmu Hukum (id=1) ===
[Ilmu Hukum] Scraping halaman 285 -> https://pta.trunojoyo.ac.id/c_search/byprod/1/285
[Ilmu Hukum] Habis di halaman 284

=== Mulai scrape prodi Magister Ilmu Hukum (id=24) ===
[Magister Ilmu Hukum] Scraping halaman 15 -> https://pta.trunojoyo.ac.id/c_search/byprod/24/15
[Magister Ilmu Hukum] Habis di halaman 14

=== Mulai scrape prodi Teknologi Industri Pertanian (id=2) ===
[Teknologi Industri Pertanian] Scraping halaman 115 -> https://pta.trunojoyo.ac.id/c_search/byprod/2/115
[Teknologi Industri Pertanian] Habis di halaman 114

=== Mulai scrape prodi Agribisnis (id=3) ===
[Agribisnis] Scraping halaman 111 -> https://pta.trunojoyo.ac.id/c_search/byprod/3/111
[Agribisnis] Habis di halaman 110

=== Mulai scrape prodi Agroteknologi (id=4) ===
[Agroteknologi] Scraping halaman 117 -> https://pta.trunojoyo.ac.id/c_search/byprod/4/117
[Agroteknologi] Habis di halaman 116

=== Mulai scrape prodi Ilmu Kelautan (id=5) ===
[Ilmu Kelautan] Scraping ha

,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak_id,abstrak_en,prodi,url
0,Dyah Ayu Citra Seza,Implementasi Fungsi Legislasi Dewan Perwakilan...,"Yudi Widagdo Harimurti, SH., MH","Safi', SH., MH",ABSTRAK\n\n Implementasi Fungsi Legislas...,ABSTRACT\n Implementation of Legislation...,Ilmu Hukum,https://pta.trunojoyo.ac.id/welcome/detail/080...
1,Maulina Nurlaily,Pertanggungjawaban Pidana Direksi BUMN (Perser...,"Tolib Effendi, SH., MH.","Dr. Eni Suastuti, SH., Mhum.",Badan Usaha Milik Negara (BUMN) adalah Badan u...,State Owned Enterprises (SOEs) are business en...,Ilmu Hukum,https://pta.trunojoyo.ac.id/welcome/detail/080...
2,Moh. Samsul Hidayat,Analisis Terhadap Kekosongan Hukum dalam Penga...,"Tolib Effendi, SH., MH.","Agus Ramdlany, SH., MH.",Kasus narkoba tidak henti-hentinya terdengar d...,"Drug cases endlessly heard on television, radi...",Ilmu Hukum,https://pta.trunojoyo.ac.id/welcome/detail/070...
3,TOMMY ADITYA PARLINDUNGAN MARBUN,PERLINDUNGAN HUKUM BAGI KONSUMEN ATAS PRODUK E...,"DR. DJULAEKA, S.H., M.HUM","DR.USWATUN HASANAH, S.H., M. HUM",Produk elektronik adalah suatu benda bergerak ...,Electronic products is an object moves through...,Ilmu Hukum,https://pta.trunojoyo.ac.id/welcome/detail/090...
4,RICA YENA IMADHORA,TELAAH KRITIS TENTANG ALASAN HUKUM YANG DIGUN...,"Dr. DENI SBY, S. H., M. S.","SAIFUL ABDULLAH, S. H., M. H.",NaN,NaN,Ilmu Hukum,https://pta.trunojoyo.ac.id/welcome/detail/070...
...,...,...,...,...,...,...,...,...
13903,Zaqiyah,PROFIL BERPIKIR KRITIS DITINJAU DARI MOTIVASI ...,"Puji Rahayu Ningsih., S.Pd.,M.Pd.","Muchamad Arif.,S.Pd.,M.Pd",Abstrak\nKemampuan berpikir kritis merupakan s...,ABSTRACT\n\nCritical thinking skill is importa...,Pendidikan Informatika,https://pta.trunojoyo.ac.id/welcome/detail/150...
13904,Irnando Arkadiantika,PENGEMBANGAN MEDIA PEMBELAJARAN VIRTUAL REALIT...,"Wanda Ramansyah, S.Pd., M.Pd.","Muhamad Afif Effindi, S.Kom., M.T.",Proses pembelajaran yang baik haruslah memuat ...,Good learning process must contain interactive...,Pendidikan Informatika,https://pta.trunojoyo.ac.id/welcome/detail/150...
13905,Mohamad Ridwan Ghozali,Pengembangan Media Pembelajaran Fisika Berbasi...,"Sigit Dwi Saputro, S.Pd., M.Pd.","Muchamad Arif, S.Pd., M.Pd.",Matakuliah Fisika Kejuruan merupakan matakulia...,Vocational Physics courses are compulsory subj...,Pendidikan Informatika,https://pta.trunojoyo.ac.id/welcome/detail/150...
13906,Ana fadzila putri,"PENGARUH MODEL PEMBELAJARAN ARIAS (ASSURANCE, ...","Ariesta Kartika Sari, S. Si., M. Pd.","Wanda ramansyah, S. Pd., M. Pd.",Penelitian ini bertujuan untuk mengetahui peng...,Based on the data values ??of class X SMK Agun...,Pendidikan Informatika,https://pta.trunojoyo.ac.id/welcome/detail/150...


[Lanjut ke Crawling_berita](Crawling_berita_PPW.ipynb)
